# Fine-Tuning and Applications With Hugging Face

## What We Are Going To Do

The first notebook built a tiny GPT from scratch.
The second notebook explained encoder-decoder transformers for seq2seq tasks.
This notebook shifts to the modern workflow students will use most often in practice:

- start from a pretrained transformer
- adapt it to a downstream task
- evaluate it
- turn it into an application

## Tasks Covered

- BERT-family sequence classification on IMDb
- GPT-style dialogue generation on DailyDialog
- summarization and chatbot inference using Hugging Face models

## Learning Outcomes

By the end of the notebook, students should be able to explain:

- the difference between bidirectional and causal attention
- how fine-tuning differs from training from scratch
- how Hugging Face datasets, tokenizers, data collators, and trainers fit together
- how to interpret logits, probabilities, and generated text in downstream applications


# Environment Setup

This notebook installs the shared environment for the whole teaching sequence.
The repository bootstrap installs from the shared `requirements.txt`, which is configured
to pull the CUDA-enabled PyTorch wheel used for this Windows teaching environment.

In the code cell below we:

- locate the project root
- run the repository bootstrap script
- print progress so students can see the environment being prepared


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bootstrap_script = PROJECT_ROOT / "scripts" / "bootstrap_env.py"

print(f"Project root: {PROJECT_ROOT}")
print(f"Running bootstrap script: {bootstrap_script}")
subprocess.check_call([sys.executable, str(bootstrap_script)])
print("Environment ready. If PyTorch was replaced during bootstrap, restart the kernel once before continuing.")


## Imports and Shared Helpers

We are now using the higher-level Hugging Face training stack for multiple tasks.
The theory remains the same, but the abstractions let us focus on task design instead of boilerplate.

In the code cell below we:

- import the libraries needed for classification and generation
- set reproducible seeds
- prepare the checkpoint directory for task-specific models


In [ ]:
import inspect
import random
from pathlib import Path

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import DatasetDict, load_dataset
from IPython.display import display
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CHECKPOINT_DIR = PROJECT_ROOT / "artifacts" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")


## Configuration

This notebook includes two fine-tuning sections.
The BERT-style classifier is intended to run in class.
The GPT-style dialogue model is included as a teaching extension and can be skipped or loaded from checkpoint.

In the code cell below we:

- choose pretrained checkpoints for each task
- choose classroom subset sizes
- define checkpoint output folders
- display the active plan


In [ ]:
BERT_CHECKPOINT = "distilbert/distilbert-base-uncased"
GPT_CHECKPOINT = "distilbert/distilgpt2"
T5_FALLBACK_CHECKPOINT = "google-t5/t5-small"

IMDB_TRAIN_SAMPLES = 3500
IMDB_VALIDATION_SAMPLES = 1000
DAILYDIALOG_TRAIN_SAMPLES = 2200
DAILYDIALOG_VALIDATION_SAMPLES = 400

RUN_GPT_FINE_TUNING = False
USE_EXISTING_CHECKPOINT_IF_AVAILABLE = True

IMDB_OUTPUT_DIR = CHECKPOINT_DIR / "distilbert_imdb_demo"
GPT_OUTPUT_DIR = CHECKPOINT_DIR / "distilgpt2_dailydialog_demo"

config_table = pd.DataFrame(
    [
        ("bert_checkpoint", BERT_CHECKPOINT),
        ("gpt_checkpoint", GPT_CHECKPOINT),
        ("imdb_train_samples", IMDB_TRAIN_SAMPLES),
        ("imdb_validation_samples", IMDB_VALIDATION_SAMPLES),
        ("dailydialog_train_samples", DAILYDIALOG_TRAIN_SAMPLES),
        ("dailydialog_validation_samples", DAILYDIALOG_VALIDATION_SAMPLES),
        ("run_gpt_fine_tuning", RUN_GPT_FINE_TUNING),
        ("imdb_output_dir", str(IMDB_OUTPUT_DIR)),
        ("gpt_output_dir", str(GPT_OUTPUT_DIR)),
    ],
    columns=["setting", "value"],
)
display(config_table)


## BERT vs GPT: Bidirectional vs Causal Attention

Before we fine-tune anything, it helps to compare the attention patterns conceptually.

- BERT-style encoders can attend in both directions
- GPT-style decoders can only attend to the left

In the code cell below we:

- build a small token sequence
- draw a bidirectional attention mask
- draw a causal attention mask
- summarize the architectural differences in a table


In [ ]:
comparison_tokens = ["The", "movie", "was", "surprisingly", "good"]
bidirectional_mask = np.ones((len(comparison_tokens), len(comparison_tokens)))
causal_mask = np.tril(np.ones((len(comparison_tokens), len(comparison_tokens))))

architecture_df = pd.DataFrame(
    [
        ("BERT family", "encoder", "bidirectional", "classification, retrieval, embeddings"),
        ("GPT family", "decoder", "causal", "generation, chat, completion"),
        ("T5 family", "encoder-decoder", "both + cross-attention", "summarization, translation"),
    ],
    columns=["model family", "core block", "attention pattern", "common tasks"],
)
display(architecture_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    bidirectional_mask,
    annot=True,
    cmap="Blues",
    cbar=False,
    xticklabels=comparison_tokens,
    yticklabels=comparison_tokens,
    ax=axes[0],
)
axes[0].set_title("Bidirectional mask (BERT-style)")
axes[0].set_xlabel("Visible tokens")
axes[0].set_ylabel("Current token")

sns.heatmap(
    causal_mask,
    annot=True,
    cmap="Reds",
    cbar=False,
    xticklabels=comparison_tokens,
    yticklabels=comparison_tokens,
    ax=axes[1],
)
axes[1].set_title("Causal mask (GPT-style)")
axes[1].set_xlabel("Visible tokens")
axes[1].set_ylabel("Current token")
plt.tight_layout()
plt.show()


## Step 1: Load and Inspect IMDb for Classification

IMDb is a classic sentiment dataset with movie reviews labeled as positive or negative.
It is a good fine-tuning dataset because the task is intuitive and the labels are easy to interpret.

In the code cell below we:

- load the dataset
- create classroom-sized train and validation subsets
- visualize label balance and review length


In [ ]:
imdb = load_dataset("stanfordnlp/imdb")

small_imdb = DatasetDict(
    train=imdb["train"].shuffle(seed=SEED).select(range(min(IMDB_TRAIN_SAMPLES, len(imdb["train"])))),
    validation=imdb["test"].shuffle(seed=SEED).select(range(min(IMDB_VALIDATION_SAMPLES, len(imdb["test"])))),
)

label_names = {0: "negative", 1: "positive"}
label_counts = pd.Series(small_imdb["train"]["label"]).map(label_names).value_counts()
review_lengths = pd.Series([len(text.split()) for text in small_imdb["train"]["text"]])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
label_counts.plot(kind="bar", color=["firebrick", "seagreen"], ax=axes[0])
axes[0].set_title("IMDb label balance")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

sns.histplot(review_lengths, bins=35, color="steelblue", ax=axes[1])
axes[1].set_title("Review length distribution")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Number of reviews")
plt.tight_layout()
plt.show()


## Step 2: Tokenize the Reviews

Fine-tuning a pretrained model means using the tokenizer that matches the checkpoint.
For sequence classification, the tokenizer creates:

- `input_ids`
- `attention_mask`

In the code cell below we:

- load the DistilBERT tokenizer
- tokenize the train and validation subsets
- visualize sequence lengths and one sample attention mask


In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT)

def tokenize_reviews(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=256)

tokenized_imdb = small_imdb.map(tokenize_reviews, batched=True)
bert_lengths = pd.Series([len(ids) for ids in tokenized_imdb["train"]["input_ids"]])

sample_review_encoding = bert_tokenizer(
    small_imdb["train"][0]["text"],
    truncation=True,
    max_length=48,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.histplot(bert_lengths, bins=30, color="purple", ax=axes[0])
axes[0].set_title("BERT tokenized review lengths")
axes[0].set_xlabel("Tokens")

sns.heatmap(
    np.array(sample_review_encoding["attention_mask"]).reshape(1, -1),
    cmap="Greens",
    cbar=False,
    ax=axes[1],
)
axes[1].set_title("Attention mask for one padded/truncated example")
axes[1].set_xlabel("Token position")
axes[1].set_yticks([])
plt.tight_layout()
plt.show()


## Step 3: Look at Real BERT Attention

The conceptual mask earlier was hand-made.
Here we inspect actual attention weights from a pretrained classifier backbone.

In the code cell below we:

- load DistilBERT with attention outputs enabled
- run one short example through the model
- visualize one attention head so students can see bidirectional connectivity


In [ ]:
bert_attention_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT,
    num_labels=2,
    output_attentions=True,
).to(DEVICE)
bert_attention_model.eval()

short_text = "This movie was surprisingly thoughtful, funny, and well acted."
short_inputs = bert_tokenizer(
    short_text,
    return_tensors="pt",
    truncation=True,
    max_length=24,
).to(DEVICE)

with torch.no_grad():
    bert_outputs = bert_attention_model(**short_inputs)

bert_tokens = bert_tokenizer.convert_ids_to_tokens(short_inputs["input_ids"][0])
bert_attention = bert_outputs.attentions[0][0, 0].detach().cpu().numpy()
plot_len = min(18, len(bert_tokens))

plt.figure(figsize=(8, 6))
sns.heatmap(
    bert_attention[:plot_len, :plot_len],
    cmap="viridis",
    xticklabels=bert_tokens[:plot_len],
    yticklabels=bert_tokens[:plot_len],
)
plt.title("One DistilBERT attention head")
plt.xlabel("Visible tokens")
plt.ylabel("Current token")
plt.tight_layout()
plt.show()


## Step 4: Prepare the IMDb Fine-Tuning Pipeline

This section builds the training loop around the pretrained classifier.
The underlying theory is the same as before: logits go into cross-entropy loss,
gradients update the weights, and validation metrics tell us whether the model improves.

In the code cell below we:

- define compatibility helpers for `TrainingArguments` and `Trainer`
- create a dynamic padding collator
- define the accuracy metric


In [ ]:
imdb_accuracy = evaluate.load("accuracy")
classification_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

def make_training_args(**kwargs):
    parameters = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" not in parameters and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    return TrainingArguments(**kwargs)

def trainer_processor_argument(processor):
    parameters = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in parameters:
        return {"processing_class": processor}
    return {"tokenizer": processor}

def compute_classification_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return imdb_accuracy.compute(predictions=predictions, references=labels)

print("Classification fine-tuning helpers are ready.")


## Step 5: Fine-Tune DistilBERT or Load a Saved Checkpoint

Fine-tuning is much faster than training from scratch because we start from useful pretrained weights.

In the code cell below we:

- load or fine-tune the classifier
- save the best local checkpoint
- plot the training and evaluation curves


In [ ]:
bert_history = None

if IMDB_OUTPUT_DIR.exists() and USE_EXISTING_CHECKPOINT_IF_AVAILABLE:
    classifier_model = AutoModelForSequenceClassification.from_pretrained(IMDB_OUTPUT_DIR).to(DEVICE)
    bert_tokenizer = AutoTokenizer.from_pretrained(IMDB_OUTPUT_DIR)
    print(f"Loaded fine-tuned classifier from {IMDB_OUTPUT_DIR}")
else:
    classifier_model = AutoModelForSequenceClassification.from_pretrained(
        BERT_CHECKPOINT,
        num_labels=2,
        id2label={0: "NEGATIVE", 1: "POSITIVE"},
        label2id={"NEGATIVE": 0, "POSITIVE": 1},
    ).to(DEVICE)

    training_args = make_training_args(
        output_dir=str(IMDB_OUTPUT_DIR),
        max_steps=250,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        logging_steps=25,
        save_total_limit=2,
        fp16=(DEVICE == "cuda"),
        report_to="none",
    )

    trainer = Trainer(
        model=classifier_model,
        args=training_args,
        train_dataset=tokenized_imdb["train"],
        eval_dataset=tokenized_imdb["validation"],
        data_collator=classification_collator,
        compute_metrics=compute_classification_metrics,
        **trainer_processor_argument(bert_tokenizer),
    )

    trainer.train()
    trainer.save_model()
    bert_tokenizer.save_pretrained(IMDB_OUTPUT_DIR)
    bert_history = pd.DataFrame(trainer.state.log_history)
    print(f"Saved fine-tuned classifier to {IMDB_OUTPUT_DIR}")

if bert_history is not None and not bert_history.empty:
    plt.figure(figsize=(10, 5))
    if "loss" in bert_history:
        loss_rows = bert_history.dropna(subset=["loss"])
        plt.plot(loss_rows["step"], loss_rows["loss"], label="train loss")
    if "eval_loss" in bert_history:
        eval_rows = bert_history.dropna(subset=["eval_loss"])
        plt.plot(eval_rows["step"], eval_rows["eval_loss"], label="eval loss")
    plt.title("DistilBERT fine-tuning curve on IMDb")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Step 6: Run Classification Inference and Inspect Probabilities

A classification model outputs logits for each class.
Applying softmax turns those logits into probabilities.

In the code cell below we:

- run the fine-tuned classifier on custom reviews
- compute softmax probabilities
- visualize how confident the model is about each label


In [ ]:
classifier_model.eval()

demo_reviews = [
    "This was a sharp, well paced film with excellent performances.",
    "The plot was messy, the acting was flat, and I wanted it to end.",
    "I expected nonsense, but it was weirdly charming and sincere.",
]

encoded_batch = bert_tokenizer(
    demo_reviews,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128,
).to(DEVICE)

with torch.no_grad():
    logits = classifier_model(**encoded_batch).logits
    probabilities = torch.softmax(logits, dim=-1).cpu().numpy()

probability_df = pd.DataFrame(
    probabilities,
    columns=["negative_probability", "positive_probability"],
)
probability_df.insert(0, "review", demo_reviews)
display(probability_df)

plt.figure(figsize=(10, 5))
x_positions = np.arange(len(demo_reviews))
plt.bar(x_positions - 0.15, probabilities[:, 0], width=0.3, label="negative")
plt.bar(x_positions + 0.15, probabilities[:, 1], width=0.3, label="positive")
plt.xticks(x_positions, [f"Review {idx + 1}" for idx in range(len(demo_reviews))])
plt.title("Classifier confidence on custom examples")
plt.ylabel("Probability")
plt.legend()
plt.tight_layout()
plt.show()


## Step 7: Load and Format DailyDialog for GPT-Style Generation

For a chatbot-style causal language model, we flatten dialogues into a single text stream.
The model learns to predict the next token in that stream, which means it implicitly learns reply patterns.

In the code cell below we:

- load DailyDialog
- format each conversation as alternating speaker turns
- create classroom-sized train and validation subsets
- visualize dialogue length statistics


In [ ]:
dailydialog = load_dataset("li2017dailydialog/daily_dialog")
dialog_column = "dialog" if "dialog" in dailydialog["train"].column_names else "dialogue"

def format_dialogue(example):
    turns = example[dialog_column]
    speaker_turns = [f"Speaker {(idx % 2) + 1}: {turn}" for idx, turn in enumerate(turns)]
    formatted = " <eos> ".join(speaker_turns)
    prompt = " <eos> ".join(speaker_turns[:-1]) if len(speaker_turns) > 1 else speaker_turns[0]
    target_reply = speaker_turns[-1]
    example["formatted_dialogue"] = formatted
    example["prompt_text"] = prompt
    example["target_reply"] = target_reply
    return example

formatted_dailydialog = dailydialog.map(format_dialogue)
small_dialog = DatasetDict(
    train=formatted_dailydialog["train"].shuffle(seed=SEED).select(
        range(min(DAILYDIALOG_TRAIN_SAMPLES, len(formatted_dailydialog["train"])))
    ),
    validation=formatted_dailydialog["validation"].shuffle(seed=SEED).select(
        range(min(DAILYDIALOG_VALIDATION_SAMPLES, len(formatted_dailydialog["validation"])))
    ),
)

turn_counts = pd.Series([text.count("<eos>") + 1 for text in small_dialog["train"]["formatted_dialogue"]])
dialogue_word_lengths = pd.Series([len(text.split()) for text in small_dialog["train"]["formatted_dialogue"]])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(turn_counts, bins=25, color="teal", ax=axes[0])
axes[0].set_title("Turns per dialogue")
axes[0].set_xlabel("Turns")

sns.histplot(dialogue_word_lengths, bins=30, color="darkgoldenrod", ax=axes[1])
axes[1].set_title("Dialogue length in words")
axes[1].set_xlabel("Words")
plt.tight_layout()
plt.show()


## Step 8: Tokenize Dialogue Data for a GPT-Style Model

GPT models are causal language models.
During fine-tuning we feed full dialogue strings and ask the model to predict every next token.

In the code cell below we:

- load the GPT tokenizer
- tokenize the formatted dialogues
- inspect token lengths
- show one tokenized example


In [ ]:
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_CHECKPOINT)
if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

def tokenize_dialogues(batch):
    return gpt_tokenizer(
        batch["formatted_dialogue"],
        truncation=True,
        max_length=256,
    )

tokenized_dialog = small_dialog.map(
    tokenize_dialogues,
    batched=True,
    remove_columns=small_dialog["train"].column_names,
)

dialog_token_lengths = pd.Series([len(ids) for ids in tokenized_dialog["train"]["input_ids"]])
sample_dialog_tokens = gpt_tokenizer.convert_ids_to_tokens(tokenized_dialog["train"][0]["input_ids"][:40])

print("Sample tokenized dialogue pieces:")
print(sample_dialog_tokens)

plt.figure(figsize=(10, 5))
sns.histplot(dialog_token_lengths, bins=30, color="slateblue")
plt.title("Token lengths for GPT dialogue fine-tuning samples")
plt.xlabel("Tokens")
plt.ylabel("Number of dialogues")
plt.tight_layout()
plt.show()


## Step 9: Visualize Real GPT Attention

This section mirrors the BERT attention inspection, but now for a causal model.
The goal is to make the left-to-right constraint visible in real model outputs.

In the code cell below we:

- load DistilGPT2 with attention outputs enabled
- run a short prompt through the model
- visualize one attention head


In [ ]:
gpt_attention_model = AutoModelForCausalLM.from_pretrained(
    GPT_CHECKPOINT,
    output_attentions=True,
).to(DEVICE)
gpt_attention_model.eval()

prompt_for_attention = "Speaker 1: Are you free tonight? <eos> Speaker 2:"
prompt_inputs = gpt_tokenizer(
    prompt_for_attention,
    return_tensors="pt",
    truncation=True,
    max_length=32,
).to(DEVICE)

with torch.no_grad():
    gpt_outputs = gpt_attention_model(**prompt_inputs)

gpt_tokens = gpt_tokenizer.convert_ids_to_tokens(prompt_inputs["input_ids"][0])
gpt_attention = gpt_outputs.attentions[0][0, 0].detach().cpu().numpy()
plot_len = min(20, len(gpt_tokens))

plt.figure(figsize=(8, 6))
sns.heatmap(
    gpt_attention[:plot_len, :plot_len],
    cmap="magma",
    xticklabels=gpt_tokens[:plot_len],
    yticklabels=gpt_tokens[:plot_len],
)
plt.title("One DistilGPT2 attention head")
plt.xlabel("Visible earlier tokens")
plt.ylabel("Current token")
plt.tight_layout()
plt.show()


## Step 10: Optionally Fine-Tune a GPT-Style Dialogue Model

This section is deliberately optional because it is the first thing to skip if class time is short.
It is still pedagogically useful because it shows that the same Trainer workflow also works for causal LM tasks.

In the code cell below we:

- define a language-modeling data collator
- either load a saved GPT checkpoint or run a short fine-tuning session
- plot the learning curve when training happens live


In [ ]:
gpt_history = None
lm_collator = DataCollatorForLanguageModeling(tokenizer=gpt_tokenizer, mlm=False)

if GPT_OUTPUT_DIR.exists() and USE_EXISTING_CHECKPOINT_IF_AVAILABLE:
    dialogue_model = AutoModelForCausalLM.from_pretrained(GPT_OUTPUT_DIR).to(DEVICE)
    gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_OUTPUT_DIR)
    print(f"Loaded GPT dialogue checkpoint from {GPT_OUTPUT_DIR}")
elif RUN_GPT_FINE_TUNING:
    dialogue_model = AutoModelForCausalLM.from_pretrained(GPT_CHECKPOINT).to(DEVICE)

    gpt_training_args = make_training_args(
        output_dir=str(GPT_OUTPUT_DIR),
        max_steps=180,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        weight_decay=0.01,
        eval_strategy="steps",
        eval_steps=45,
        save_steps=45,
        logging_steps=20,
        save_total_limit=2,
        fp16=(DEVICE == "cuda"),
        report_to="none",
    )

    gpt_trainer = Trainer(
        model=dialogue_model,
        args=gpt_training_args,
        train_dataset=tokenized_dialog["train"],
        eval_dataset=tokenized_dialog["validation"],
        data_collator=lm_collator,
        **trainer_processor_argument(gpt_tokenizer),
    )

    gpt_trainer.train()
    gpt_trainer.save_model()
    gpt_tokenizer.save_pretrained(GPT_OUTPUT_DIR)
    gpt_history = pd.DataFrame(gpt_trainer.state.log_history)
    print(f"Saved GPT dialogue checkpoint to {GPT_OUTPUT_DIR}")
else:
    dialogue_model = AutoModelForCausalLM.from_pretrained(GPT_CHECKPOINT).to(DEVICE)
    print("Skipping GPT fine-tuning in this run. Using the pretrained checkpoint for generation demos.")

if gpt_history is not None and not gpt_history.empty:
    plt.figure(figsize=(10, 5))
    if "loss" in gpt_history:
        loss_rows = gpt_history.dropna(subset=["loss"])
        plt.plot(loss_rows["step"], loss_rows["loss"], label="train loss")
    if "eval_loss" in gpt_history:
        eval_rows = gpt_history.dropna(subset=["eval_loss"])
        plt.plot(eval_rows["step"], eval_rows["eval_loss"], label="eval loss")
    plt.title("GPT-style dialogue fine-tuning curve")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Step 11: Build Simple Applications

A notebook on LLMs should end with applications students recognize.
We will do two:

- a chatbot-style next-reply generator
- a summarizer for a conversation

In the code cell below we:

- generate a reply from a dialogue prompt using GPT-style decoding
- summarize a conversation with a T5 model
- compare decoding settings for the chatbot output


In [ ]:
dialogue_model.eval()

chat_prompt = "Speaker 1: Are you still coming to dinner? <eos> Speaker 2:"
chat_inputs = gpt_tokenizer(chat_prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    greedy_reply_ids = dialogue_model.generate(
        **chat_inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=gpt_tokenizer.eos_token_id,
    )
    sampled_reply_ids = dialogue_model.generate(
        **chat_inputs,
        max_new_tokens=40,
        do_sample=True,
        top_k=40,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=gpt_tokenizer.eos_token_id,
    )

greedy_reply = gpt_tokenizer.decode(greedy_reply_ids[0], skip_special_tokens=True)
sampled_reply = gpt_tokenizer.decode(sampled_reply_ids[0], skip_special_tokens=True)

summarizer_source = CHECKPOINT_DIR / "t5_samsum_demo"
summarizer_checkpoint = str(summarizer_source if summarizer_source.exists() else T5_FALLBACK_CHECKPOINT)
summarizer_tokenizer = AutoTokenizer.from_pretrained(summarizer_checkpoint)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_checkpoint).to(DEVICE)
summarizer_model.eval()

conversation_for_summary = (
    "Ava: We moved the project meeting to 3 PM.\n"
    "Liam: That works for me.\n"
    "Ava: Please bring the updated slides.\n"
    "Liam: I will also send the draft before lunch."
)

summary_inputs = summarizer_tokenizer(
    "summarize: " + conversation_for_summary,
    return_tensors="pt",
    truncation=True,
    max_length=256,
).to(DEVICE)

with torch.no_grad():
    summary_ids = summarizer_model.generate(
        **summary_inputs,
        max_new_tokens=50,
        do_sample=False,
    )

summary_text = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

application_df = pd.DataFrame(
    [
        ("Chatbot reply - greedy", greedy_reply),
        ("Chatbot reply - sampled", sampled_reply),
        ("Conversation summary", summary_text),
    ],
    columns=["application", "output"],
)
display(application_df)


## Wrap-Up

Across the three notebooks, students have now seen the whole story:

- how a decoder-only transformer is built from scratch
- why attention is necessary for seq2seq tasks
- how encoder-decoder models use cross-attention as memory retrieval
- how pretrained transformers are fine-tuned for classification, summarization, and dialogue

That covers the conceptual pipeline from raw text and attention math all the way to modern Hugging Face applications.
